In [3]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


default_proxy_config = {
    'http': 'http://127.0.0.1:7890',
    'https': 'http://127.0.0.1:7890',
    'all': 'socks5://127.0.0.1:7890',
}


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [14]:
import json
import requests

venues = ['ICLR 2024 oral', 'ICLR 2024 poster', 'ICLR 2024 spotlight', 'Submitted to ICLR 2024']
details = ['replyCount,presentation']
domains = ['ICLR.cc/2024/Conference']
limits = [1001]

for venue in venues:
    for detail in details:
        for domain in domains:
            for limit in limits:
                offset = 2000
                resp = requests.get(f'https://api2.openreview.net/notes?content.venue={venue}&details={detail}&domain={domain}&limit={limit}&offset={offset}')
                break

print(resp, resp.text)
print(resp.url)
# json.dump(resp.json(), open('limit.json', 'w'), indent=4, ensure_ascii=False)

<Response [400]> {"name":"ValidationError","message":"limit must be <= 1000","status":400,"details":{"path":"limit","comparison":"<=","limit":1000,"invalidValue":1001,"reqId":"2024-01-31-971781"}}
https://api2.openreview.net/notes?content.venue=Submitted%20to%20ICLR%202024&details=replyCount,presentation&domain=ICLR.cc/2024/Conference&limit=1001&offset=2000


In [6]:
def _cell_analysis_1():
    """
        we can get gallery ids at this step, from some specific author
    """
    import requests
    resp = requests.get('https://ltn.hitomi.la/artist/mutou%20mato-all.nozomi', proxies=default_proxy_config)
    print(resp)
    print(resp.content)
    print(resp.headers['content-type'])

    def decode_gallery_id(resp):
        bytes = resp.content
        assert len(bytes) % 4 == 0, f'length of bytes {len(bytes)} not divisible by 4'
        ids = []
        for i in range(0, len(bytes), 4):
            ids.append(int.from_bytes(bytes[i:i+4], 'big'))
        return ids

    print(decode_gallery_id(resp))

_cell_analysis_1()

<Response [200]>
b'\x00*\x9a\x7f\x00*\x98\xf9\x00*\x91\xd7\x00*z\x00\x00*F\xe5\x00)\xd6\xed\x00)\xb7l\x00)\xae\xc8\x00)\x97\x8c\x00)\x96\x9a\x00)U\xad\x00)I<\x00)G5\x00)\x12\x80\x00(\xccu\x00(\xaa\\\x00(\xa0\xff\x00(\x9eb\x00\'\x01\x15\x00&\xd4\xc7\x00&\x84\x16\x00&S\x8c\x00&L\x96\x00%\x7f\xed\x00$\xdd\xa5\x00$Q\xdf\x00$\x19\xea\x00$\x0ej\x00#\xf4\x10\x00#\xdc\xf4\x00#\xb4P\x00#\x92#\x00#\x8a;\x00",\xf5\x00!\xe1^\x00!\xc8W\x00!h\x86\x00!M9\x00!@b\x00!,!\x00 \xcf&\x00 \xc1\x0f\x00 \xa6E\x00 F\x03\x00 >\xed\x00 80\x00 -K\x00 \x18e\x00 \x13\x17\x00\x1f\xa9a\x00\x1f\x08\xef\x00\x1e\xa4\xf7\x00\x1e\x8b\xdf\x00\x1eU\xb4\x00\x1eR\x9b\x00\x1eM\xdf\x00\x10\x0b\xff\x00\x1d\xe2\xce\x00\x1d\xd9V\x00\x1d\xd9A\x00\x1d\xb8\xc3\x00\x1d\x8b\xbc\x00\x1d\x0ex\x00\x1d\x0ep\x00\x1d\r\xc2\x00\x1d\r\xb0\x00\x1c\xc5\xd9\x00\x1c\xc5W\x00\x1c\xadv\x00\x1c9\r\x00\x1b\xcdB\x00\x1b\xc4\x06\x00\x1b\xba\x8a\x00\x1b\x80\xe8\x00\x1bgd\x00\x1bc\t\x00\x1ah\xd3\x00\x1a`/\x00\x1a]9\x00\x1a8\xf6\x00\x1a8\xf4\x00\x19\xd2j\x

In [40]:
def _cell_analysis_2():
    """
        we can get gallery links, names and metadata(tags, language, type, series, author) at this step
    """
    import urllib.parse as urlparse
    import requests
    resp = requests.get('https://ltn.hitomi.la/galleryblock/2813534.html', proxies=default_proxy_config, headers={'referer': 'https://ltn.hitomi.la/artist/mutou%20mato-all.nozomi'})
    print(resp)
    print(resp.content)
    with open('ex.html', 'wb') as f:
        f.write(resp.content)
    print()

    from RFC.utils.parse import (
        BeautifulSoup,
        parse,
    )
    
    def parse_gallery_block(resp):
        parse_config = {
            'all_links': {
                ('result', 'href', (None,)): {},
            },
        }
        return parse(BeautifulSoup(resp.content), parse_config['all_links'])

    def get_gallery_link(links):
        return urlparse.urljoin('https://hitomi.la/', links[0].attrs['href'])

    def get_metadata(links):
        metadata = {}
        for link in links:
            if link.attrs['href'].startswith('/tag/'):
                if 'tag' not in metadata:
                    metadata['tag'] = []
                metadata['tag'].append(link.text)
            if link.attrs['href'].startswith('/series/'):
                if 'series' not in metadata:
                    metadata['series'] = []
                metadata['series'].append(link.text)
            if link.attrs['href'].startswith('/type/'):
                if 'type' not in metadata:
                    metadata['type'] = []
                metadata['type'].append(link.text)
            if link.attrs['href'].startswith('/artist/'):
                if 'artist' not in metadata:
                    metadata['artist'] = []
                metadata['artist'].append(link.text)
            if link.attrs['href'].startswith('/index-'):
                if 'language' not in metadata:
                    metadata['language'] = []
                metadata['language'].append(link.text)
        return metadata


    links = parse_gallery_block(resp)
    gallery_link = get_gallery_link(links)
    metadata = get_metadata(links)
    
    import json
    print(gallery_link, json.dumps(metadata, indent=4, ensure_ascii=False), sep='\n\n')

_cell_analysis_2()

<Response [200]>
b'<div class="imageset">\n<a href="/imageset/gweda--17282018--2024.01.30-\xe6\x97\xa5\xe6\x9c\xac\xe8\xaa\x9e-2813534.html" class="lillie"><div class="dj-img-cont">\n<div class="dj-img1"><picture><source class="picturelazyload" type="image/avif" data-srcset="//tn.hitomi.la/avifbigtn/3/38/55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383.avif 2x, //tn.hitomi.la/avifsmallbigtn/3/38/55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383.avif 1x"><img class="lazyload" loading="lazy" data-src="//tn.hitomi.la/webpbigtn/3/38/55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383.webp"></picture></div>\n<div class="dj-img2"><picture><source class="picturelazyload" type="image/avif" data-srcset="//tn.hitomi.la/avifbigtn/4/4d/9cf0a60c2b556ab19b5e5023174323f5300ba4de9eb955850b57bd6fe29824d4.avif 2x, //tn.hitomi.la/avifsmallbigtn/4/4d/9cf0a60c2b556ab19b5e5023174323f5300ba4de9eb955850b57bd6fe29824d4.avif 1x"><img class="lazyload" loading="lazy" da

In [67]:
import re
import time
import requests


while True:
    try:
        resp = requests.get('https://ltn.hitomi.la/gg.js', proxies=default_proxy_config, headers={'referer': 'https://hitomi.la/reader/509748.html'})
        result = re.findall('b: \'([0-9]*)\/', resp.text)
        assert len(result) == 1
        break
    except Exception as e:
        time.sleep(1)
        
resp.text

"'use strict';\ngg = { m: function(g) {\nvar o = 0;\nswitch (g) {\ncase 412:\ncase 1736:\ncase 523:\ncase 1153:\ncase 1982:\ncase 2020:\ncase 3875:\ncase 1827:\ncase 2101:\ncase 273:\ncase 3839:\ncase 1972:\ncase 2097:\ncase 2774:\ncase 2687:\ncase 3519:\ncase 416:\ncase 2239:\ncase 1161:\ncase 1406:\ncase 806:\ncase 213:\ncase 180:\ncase 3963:\ncase 3695:\ncase 457:\ncase 2910:\ncase 537:\ncase 392:\ncase 2414:\ncase 3817:\ncase 394:\ncase 3427:\ncase 3219:\ncase 437:\ncase 2480:\ncase 3581:\ncase 4015:\ncase 3662:\ncase 3343:\ncase 3461:\ncase 823:\ncase 3510:\ncase 1742:\ncase 2078:\ncase 1062:\ncase 946:\ncase 2736:\ncase 401:\ncase 775:\ncase 2329:\ncase 2005:\ncase 1649:\ncase 1874:\ncase 2003:\ncase 2975:\ncase 3767:\ncase 992:\ncase 1754:\ncase 3778:\ncase 286:\ncase 4028:\ncase 986:\ncase 2969:\ncase 1464:\ncase 1925:\ncase 1513:\ncase 2169:\ncase 3164:\ncase 2289:\ncase 730:\ncase 3138:\ncase 3971:\ncase 613:\ncase 3724:\ncase 1248:\ncase 1140:\ncase 218:\ncase 3233:\ncase 16

In [78]:
import re
import json
import time
import pickle
import requests

def _request_prefix():
    while True:
        try:
            resp = requests.get('https://ltn.hitomi.la/gg.js', proxies=default_proxy_config, headers={'referer': 'https://hitomi.la/reader/509748.html'})
            prefix = re.findall('b: \'([0-9]*)\/', resp.text)
            value_o = [int(x) for x in re.findall('o = ([0-1])', resp.text)]
            hit_hash = [int(x) for x in re.findall('case ([0-9]+):', resp.text)]
            assert len(prefix) == 1 and len(value_o) == 2
            return prefix[0], value_o, hit_hash
        except Exception as e:
            time.sleep(1)
    
def _from_hash_to_url(hash, prefix):
    return f'{prefix}/{int(hash[-1]+hash[-3:-1], 16)}/{hash}'
    # '1706601602/2151/b5ddc3da968890e7ef4b94982e286a8cd27319c9fbc4853012d5be6546ebc678'
    
def _get_base(value_o, hit_hash, hash):
    o = value_o[-1] if int(hash[-1]+hash[-3:-1], 16) in hit_hash else value_o[0]
    return chr(ord('a')+o)

def _add_image(id, img_type, img_hash, prefix, base):
    url = f'https://{base}a.hitomi.la/{img_type}/{_from_hash_to_url(img_hash, prefix)}.{img_type}'
    # https://aa.hitomi.la/avif/1706626802/824/55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383.avif

result = pickle.load(open('saves/mutou mato/subs/1004701/_result.pkl', 'rb'))
# print(result)
js = result['response'].text
assert js.startswith('var galleryinfo = ')
js = json.loads(js[len('var galleryinfo = '):])
prefix, value_o, hit_hash = _request_prefix()
for id, img in enumerate(js['files']):
    hash = img['hash']
    id = f'{id:04d}'
    added = False
    base = _get_base(value_o, hit_hash, hash)
    if 'hasavif' in img and img['hasavif'] == 1:
        _add_image(id, 'avif', hash, prefix, base)
        added = True
    if 'haswebp' in img and img['haswebp'] == 1:
        _add_image(id, 'webp', hash, prefix, base)
        added = True
    if not added and 'hasjxl' in img and img['hasjxl'] == 1:  # optional
        _add_image(id, 'jxl', hash, prefix, base)
        added = True

https://aa.hitomi.la/webp/1706637601/3367/b3b76211473935d50c3cadc14f885081049b2e0ea667e1d9285eb5775153327d.webp
https://aa.hitomi.la/webp/1706637601/614/2459f8427155295e02d5da0ec3428b45fd209f8166f9bb2baf9f160b27a07662.webp
https://ba.hitomi.la/webp/1706637601/1024/b00ef04e72152ed9fe876c40985d828c943ec4714a842085b034c3831cf22004.webp
https://aa.hitomi.la/webp/1706637601/3800/b1105d0d8f7a93abbfced6bb0fa374ba53bf7e90cc4f776d62236388fa125d8e.webp
https://aa.hitomi.la/webp/1706637601/3507/935a12bdfe5e683b1f41be7e6a9789bef186f6f5a407c7aa8c3aaefc6c93ab3d.webp
https://ba.hitomi.la/webp/1706637601/402/f48e17b6386cfb9add1f2dcf68399820c92c00c3b91d4976cbccdb57f1b2e921.webp
https://ba.hitomi.la/webp/1706637601/3851/cd220c2001a80213a528179f51f78a47396ee74b7165fc827917300fa29380bf.webp
https://ba.hitomi.la/webp/1706637601/2276/284a67c0d5614f91dfa4fd1b676cb1b84320bd124a52087d67ef3da45b549e48.webp
https://ba.hitomi.la/webp/1706637601/3954/ab72a213768ffd5c5b07e56477f33d60623f24b36c0fc06b6784414ff3eeb72f

In [64]:
def _cell_analysis_2():

    import urllib.parse as urlparse
    import requests
    resp = requests.get('https://ltn.hitomi.la/galleries/2813534.js', proxies=default_proxy_config, headers={'referer': 'https://ltn.hitomi.la/artist/mutou%20mato-all.nozomi'})
    print(resp)
    print(resp.text)
    
    
_cell_analysis_2()

<Response [200]>
var galleryinfo = {"artists":[{"url":"/artist/gwegwe-all.html","artist":"gwegwe"}],"languages":[],"language":"japanese","files":[{"hasavif":1,"haswebp":1,"name":"0001.png","hash":"55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383","width":1652,"height":2679,"hasjxl":1},{"height":1558,"width":1000,"hasjxl":1,"hasavif":1,"haswebp":1,"name":"0002.png","hash":"0a656735699096ac193dab014d78fc2b98f9569a8b0e38505a7880b533a97ca6"},{"width":1242,"height":842,"hasjxl":1,"hasavif":1,"haswebp":1,"hash":"4b80f79e43ff262c2dd225f0efbae62979250402d84ef2c75367cb46c42e08b9","name":"0003.png"},{"hasavif":1,"haswebp":1,"hash":"91a5c560797a3e5bba051d646becce199281cfef00ea5dae33c398b5e07fe000","name":"0004.png","height":1104,"width":1584,"hasjxl":1},{"haswebp":1,"hasavif":1,"hash":"890e3cb2ba40741e299197fb5c3b2e6d545d264d8946b6b26692dd6d7f6a845e","name":"0005.png","height":789,"width":1144,"hasjxl":1},{"hash":"df2488bc3175c0886772610dbf09eb5ca2b691b1d6110a2779dc93ebb03b940c","

In [80]:
def from_hash_to_url(hash, prefix=1706601602):
    return f'{prefix}/{int(hash[-1]+hash[-3:-1], 16)}/{hash}'

import requests
resp = requests.get('https://ba.hitomi.la/webp/1706637601/1224/d056bc87f4ff4b496048f662a173e4fcc109bbd24782913064810f44293e7c84.webp', proxies=default_proxy_config, headers={'referer': 'https://hitomi.la/reader/2813534.html'})
print(resp)
with open('img.webp', 'wb') as f:
    f.write(resp.content)

<Response [200]>


In [1]:
# hash = '55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383'
# hash = '0a656735699096ac193dab014d78fc2b98f9569a8b0e38505a7880b533a97ca6'
# hash = 'b5ddc3da968890e7ef4b94982e286a8cd27319c9fbc4853012d5be6546ebc678'
def from_hash_to_url(hash, prefix=1706601602):
    return f'{prefix}/{int(hash[-1]+hash[-3:-1], 16)}/{hash}'


from_hash_to_url(hash)

'1706601602/2151/b5ddc3da968890e7ef4b94982e286a8cd27319c9fbc4853012d5be6546ebc678'